# 04 · Failure Analysis
**Time: 20–30 minutes.**

## Learning Objectives
Locate the first observable error, connect a video frame to logged state, and separate facts from hypotheses.

## Background
A failed rollout can arise during approach, grasp, lift, transport, release or stabilization. A video alone rarely establishes the cause. Use the state and action traces to support your interpretation. Gripper opening (meters) and normalized gripper command (unitless) use different axes.

## Step-by-step Instructions
### Step 1 — Select a real failure
Use the primary student model. If it has no completed failure, use an explicitly attributed instructor reference failure. The fallback never changes student evaluation scores. Missing reference evidence produces a clear message; no artificial failure is generated.


In [ ]:
from pathlib import Path
import os, json
from IPython.display import display, Markdown, Image, Video
from manipulation_lab.common import ROOT, DATA, write_json
from manipulation_lab.notebook import workspace, command, next_stage, stage_path
from manipulation_lab.teaching import read_json, markdown_table
GROUP = os.environ.get("LAB_GROUP", "group_01")  # Use the group assigned by your instructor.
RUN = workspace(GROUP)
EXPERIMENT = stage_path(RUN, "experiment")
experiment = read_json(EXPERIMENT / "experiment.json")
assert experiment["complete"], "Complete both learning-rate runs in Notebook 02 first."
primary = experiment["variants"][experiment["primary_variant"]]
TRAINING = EXPERIMENT / primary["training"]
EVALUATION = EXPERIMENT / primary["evaluation"]


In [ ]:
FAILURE_EVALUATION = EVALUATION
results = read_json(FAILURE_EVALUATION / "summary.json")
failures = [r for r in results["episodes"] if not r["stable_success"] and not r["error"] and r["video"]]
if not failures:
    reference = Path(os.environ.get("LAB_REFERENCE_ROOT", ROOT / "checkpoints"))
    FAILURE_EVALUATION = reference / "reference_evaluation"
    if not (FAILURE_EVALUATION / "summary.json").is_file():
        raise FileNotFoundError("No primary-policy failure and no instructor reference. Ask the instructor to run prepare_reference.py --train and set LAB_REFERENCE_ROOT; see docs/server_setup.md.")
    results = read_json(FAILURE_EVALUATION / "summary.json")
    assert results["policy_role"] == "instructor_reference"
    failures = [r for r in results["episodes"] if not r["stable_success"] and not r["error"] and r["video"]]
    if not failures:
        raise RuntimeError("No completed real failure is available. Instructor must supply attributed evidence; do not invent a case.")
    print("Evidence source: instructor reference. Student scores are unchanged.")
display(Markdown(markdown_table(failures, ["episode_id", "source_episode", "steps", "environment_success", "stable_success"])))
EPISODE = failures[0]["episode_id"]  # You may choose another failure from this table.
selected = next(r for r in failures if r["episode_id"] == EPISODE)
print("Selected:", EPISODE, "Available control steps: 0..", selected["steps"])


### Step 2 — Choose control steps and mark the first observable error
Watch the full video, then change `KEYFRAME_STEPS`. Set `FIRST_ERROR_STEP` to the first step where you can support an error from evidence; leave `None` while undecided.

State 0 precedes the first action. Action `a[t-1]` leads to state `s[t]`. Video is sampled every four steps **after** an action, so a requested state may have no exact video frame. The nearest actual frame is shown with its offset; no missing frames are synthesized. State values below belong to the **requested control step**, not to an offset frame. Rerun Steps 2–4 to save a new analysis version.


In [ ]:
display(Video(str(FAILURE_EVALUATION / selected["video"]), embed=True))
KEYFRAME_STEPS = [0, selected["steps"] // 2, selected["steps"]]
FIRST_ERROR_STEP = None  # Replace with an observed integer step; this is your annotation.


In [ ]:
from manipulation_lab.reporting import failure_evidence
ANALYSIS = next_stage(RUN, "analysis")
ANALYSIS.mkdir()
facts = failure_evidence(FAILURE_EVALUATION, EPISODE, ANALYSIS,
                         requested_steps=KEYFRAME_STEPS, first_error_step=FIRST_ERROR_STEP)
display(Image(filename=str(ANALYSIS / "failure_signals.png")))
for frame in facts["keyframes"]:
    print("Requested step:", frame["requested_control_step"], "Recorded step:", frame["recorded_control_step"],
          "Offset:", frame["offset_steps"], "Video frame:", frame["video_frame_index"])
    display(Markdown(markdown_table([frame["requested_state"]])))
    display(Image(filename=str(ANALYSIS / frame["file"])))


### Step 3 — Explain your evidence
**Q5. Identify one failure and propose a possible improvement.** Choose a stage (or `unknown`), describe the earliest visible error, propose a cause, and suggest a test of that cause. The red line is your annotation, not an automatically inferred cause.


In [ ]:
FAILURE_STAGE = ""
FIRST_OBSERVABLE_ERROR = ""
HYPOTHESIZED_CAUSE = ""
IMPROVEMENT = ""
write_json(ANALYSIS / "failure_analysis.json", {
    "episode": EPISODE, "evidence_directory": os.path.relpath(FAILURE_EVALUATION, ANALYSIS),
    "evidence_policy_role": results["policy_role"], "evidence_checkpoint_sha256": results["checkpoint_sha256"],
    "stage": FAILURE_STAGE, "first_observable_error": FIRST_OBSERVABLE_ERROR,
    "first_error_control_step": FIRST_ERROR_STEP,
    "hypothesized_cause": HYPOTHESIZED_CAUSE, "improvement": IMPROVEMENT})


### Step 4 — Generate the report and evidence archive
Complete Q1 in Notebook 01, Q2–Q3 in Notebook 02 and Q4 in Notebook 03 by rerunning their **answer cells only**, then regenerate this report. Blank answers or an unmarked error keep it in **DRAFT** status. The generated Markdown is an evidence appendix; write a concise **1–2 page report** using the [assignment guide](../docs/assignment_student.md).


In [ ]:
REPORT = next_stage(RUN, "report")
command("generate_report.py", "--run", RUN, "--output", REPORT)
from manipulation_lab.reporting import notebook_report_markdown
display(Markdown(notebook_report_markdown(REPORT / "report.md")))
ARCHIVE = RUN.parent / (RUN.name + "_" + REPORT.name + "_submission.zip")
command("export_submission.py", "--run", RUN, "--output", ARCHIVE)
print("Evidence archive:", ARCHIVE)
print("Add your own 1–2 page PDF report. Download the ZIP from Jupyter's file browser.")


## Expected Outputs
A real failed-rollout video, four separate signal axes, selected keyframes with exact requested-state values, a saved first-error annotation, a report draft/completed-answer status, and an evidence ZIP. The ZIP includes both configurations, checkpoints, curves, evaluation records, videos and class CSV rows.

## Questions for Students
Does the visible error prove your proposed cause? What additional controlled test would distinguish that cause from an alternative explanation? Include this limitation in Q5.

## Completion Checklist
- [ ] I selected real, attributed failure evidence and inspected at least three steps.
- [ ] I marked the first observable error and separated fact from hypothesis.
- [ ] I answered Q1–Q5 and regenerated the report after saving answers.
- [ ] I prepared a 1–2 page report and the evidence ZIP with both variants.
- [ ] I retained original outcomes and did not rerun seeds to improve the score.

Grading: demonstration understanding 20, training/experiment 25, evaluation 30, failure analysis/discussion 25. Success rate itself is not a grade.
